# GLiNER backend

> Route B: GLiNER2 as a native backend through the `gliner2` package: typed questions as classification heads, GLiNER2's JSONL and trainer, and its extraction features one attribute away.

In [ ]:
#| default_exp gliner

In [ ]:
#| export
import json, os, tempfile
from pathlib import Path

import numpy as np
from fastcore.basics import patch

from sysone.core import *
from sysone.data import TypedDecisions
from sysone.losses import fit_temperatures, answer_metrics

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail

Route A put GLiNER2.5-Decide's encoder under the sysone head. Route B keeps the whole model and drives it through the `gliner2` package (`sysone[gliner]`): a choice becomes a single-label head with the criteria as `{label: description}` and the instructions as the head's prompt, a noul a `["yes", "no"]` head read as P(yes), a score a head of string levels read as Σ i·pᵢ. Full distributions come from GLiNER2's constrained classifier, whose single-label heads are softmaxed, so the distribution is proper; multi-label heads are sigmoids and are never used for typed decisions. Several questions of one call share one row, GLiNER2's own batching, and answers come back in the Jev schema like every other backend.

`gliner2` 2.0.0 installs without its `local` extra and then runs on the transformers 5 stack sysone uses (its extras pin transformers below 5; the package itself does not need it), so it can share an environment with the rest of sysone; it does need `peft` even for inference. What this route gives up: the feature cache (the labels sit in the input, so encoder outputs change with the label set), `lr_find`, the presets and RLCD (GLiNER2's loss is a binary cross-entropy on the true labels, so soft targets collapse to the label), and Laya's export format.

```{mermaid}
flowchart LR
    Q["typed questions"] --> C["choice: a single-label head,<br/>the criteria as {label: description}"]
    Q --> N["noul: a [yes, no] head,<br/>read as P(yes)"]
    Q --> S["score: a head of levels,<br/>read as Σ i·pᵢ"]
    C & N & S --> SCH["one schema per call,<br/>a temperature per task"]
    SCH --> G["GLiNER2's constrained classifier:<br/>one pass over the state"]
    G --> A["answers in the Jev schema"]
```

## Questions as heads

Each kind of question becomes a GLiNER2 classification head in its own way, so `gliner_labels` is a [generic function](00_core.ipynb#generic-functions-and-multiple-dispatch) with a method per kind: a choice's labels are its keys, a score's are its levels, and a noul's are `yes` and `no`.

In [ ]:
#| export
@dispatch
def gliner_labels(q:Choice) -> tuple:
    "A choice's GLiNER2 labels (its keys), their descriptions, and the answer key each label stands for"
    labels = [str(k) for k in q.keys]
    desc = {str(k): render_criterion(v) for k, v in q.criteria.items() if v not in (None, "")}
    return labels, desc, list(q.keys)

@dispatch
def gliner_labels(q:Score) -> tuple:
    "A score's GLiNER2 labels, its levels' indices, with their descriptions"
    labels = [str(i) for i in range(q.k)]
    return labels, {str(i): render_criterion(c) for i, c in enumerate(q.criteria)}, labels

@dispatch
def gliner_labels(q:Noul) -> tuple:
    "A noul's GLiNER2 labels, yes then no, standing for true and false"
    crit = q.criteria or {}
    desc = {lab: render_criterion(crit[k]) for lab, k in (("yes", "true"), ("no", "false")) if crit.get(k) not in (None, "")}
    return ["yes", "no"], desc, ["true", "false"]

def gliner_tasks(questions, temperatures:dict|None=None) -> list:
    "One single-label task per question: name, labels (with descriptions), prompt and temperature"
    out = []
    for qid, q in as_questions(questions).items():
        labels, desc, keys = gliner_labels(q)
        out.append({"task": qid, "labels": {l: desc.get(l, "") for l in labels} if desc else labels, "instruction": q.instructions,
                    "temperature": lookup_temperature(temperatures, q.qtype, q.k), "keys": keys, "names": labels})
    return out

In [ ]:
from pathlib import Path

In [ ]:
recs = [json.loads(l) for l in Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()]
tasks = gliner_tasks(recs[0]["questions"], {"noul:2": 1.5})
[(t["task"], t["names"], t["temperature"]) for t in tasks]

[('action', ['continue', 'human_review', 'observe', 'stop'], 1.0),
 ('needs_review', ['yes', 'no'], 1.5),
 ('outcome', ['failure', 'harmful', 'partial', 'success'], 1.0),
 ('risk', ['0', '1', '2', '3'], 1.0),
 ('urgency', ['0', '1', '2', '3'], 1.0)]

In [ ]:
t = {x["task"]: x for x in tasks}
test_eq(t["needs_review"]["names"], ["yes", "no"])
test_eq(t["needs_review"]["labels"]["yes"], recs[0]["questions"]["needs_review"]["criteria"]["true"])
test_eq((t["needs_review"]["temperature"], t["action"]["temperature"]), (1.5, 1.0))
test_eq(t["risk"]["names"], ["0", "1", "2", "3"])

## GlinerDecider

`GlinerDecider.predict` builds one constrained-classifier schema for the call, a single-label task per question at its calibrated temperature (GLiNER2 applies it inside its softmax), runs it on the state's text, and reads each task's full distribution back into the question's key order. The classifier call is a function of the text and the tasks, `classify(text, tasks) -> {task: {label: p}}`; the default one uses `gliner2`, and a stand-in makes the mapping testable without it. `.model` is the loaded `AutoExtractor`, so `extract_entities`, `extract_relations`, `extract_json` and `create_schema()` are one attribute away.

In [ ]:
#| export
def gliner2_classify(model):
    "The classify function of a loaded GLiNER2 model: full distributions from its constrained classifier"
    from gliner2.classification import Classifier, ClassificationSchema
    clf = Classifier(model)
    def classify(text, tasks):
        schema = ClassificationSchema()
        for t in tasks: schema = schema.single(t["task"], t["labels"], temperature=t["temperature"], instruction=t["instruction"])
        res = clf.classify(text, schema)
        return {t["task"]: dict(res.probabilities(t["task"])) for t in tasks}
    return classify

class GlinerDecider:
    "Typed answers in the Jev schema from a GLiNER2 model"
    def __init__(self, model=None, temperatures:dict|None=None, classify=None, digits:int|None=4, name:str="gliner2"):
        self.model, self.temperatures, self.digits, self.name = model, dict(temperatures or {}), digits, name
        self.classify = classify or gliner2_classify(model)
    def __repr__(self): return f"GlinerDecider({self.name} · {len(self.temperatures)} temperatures)"

    def probabilities(self, state, questions) -> dict:
        "The option probabilities of every question, in key order"
        tasks = gliner_tasks(questions, self.temperatures)
        out = self.classify(render_state(state), tasks)
        qs, res = as_questions(questions), {}
        for t in tasks:
            p = np.array([float(out[t["task"]].get(n, 0.0)) for n in t["names"]])      # in label order
            p = p / p.sum() if p.sum() > 0 else np.full(len(p), 1 / len(p))
            at = {k: i for i, k in enumerate(t["keys"])}                              # label i answers key keys[i]
            res[t["task"]] = np.array([p[at[k]] for k in qs[t["task"]].keys])
        return res

    def predict(self, state, questions) -> dict:
        "Answers `{qid: answer}` in the Jev schema"
        qs = as_questions(questions)
        return {qid: fill_answer(qs[qid], p, self.digits) for qid, p in self.probabilities(state, questions).items()}

    @classmethod
    def load(cls, path, device=None, **kw):
        "A GLiNER2 checkpoint or a sysone GLiNER2 export (its temperatures in `sysone_gliner.json`)"
        from gliner2 import AutoExtractor
        p, meta = Path(str(path)), {}
        if (p / "sysone_gliner.json").exists(): meta = json.loads((p / "sysone_gliner.json").read_text())
        if meta.get("adapter"):
            from peft import PeftModel
            model = PeftModel.from_pretrained(AutoExtractor.from_pretrained(meta["base"]), str(p))
        else: model = AutoExtractor.from_pretrained(str(path))
        if device is not None and hasattr(model, "to"): model = model.to(device)
        return cls(model, meta.get("temperature"), name=meta.get("name", str(path)), **kw)

With a stand-in classifier that returns fixed distributions, the answers come back in the schema, a noul's `yes` read as P(true), a score's levels as its expected level:

In [ ]:
def fake_classify(text, tasks):
    return {t["task"]: {n: (i + 1) / sum(range(1, len(t["names"]) + 1)) for i, n in enumerate(t["names"])} for t in tasks}

gd = GlinerDecider(classify=fake_classify)
ans = gd.predict(recs[0]["state"], recs[0]["questions"])
ans["needs_review"], ans["risk"]["score"]

({'type': 'noul',
  'noul': 0.3333,
  'confidence': 0.6667,
  'answer_confidence': 0.6667},
 2.0)

In [ ]:
test_close(ans["needs_review"]["noul"], 1 / 3, eps=1e-4)                  # "yes" is the first of two labels: p = 1/3
test_close(ans["risk"]["score"], sum(i * (i + 1) / 10 for i in range(4)), eps=1e-4)
test_eq(ans["action"]["choice"], list(recs[0]["questions"]["action"]["criteria"])[-1])

## Training data

`gliner_example` writes one record in GLiNER2's training format: the state as `input`, and under `output.classifications` one entry per question with gold, its labels, the gold label as a one-element list (GLiNER2's trainer requires a list), the instructions as the prompt and the descriptions. Extraction annotations a record carries (`entities`, `relations`, `json_structures`) pass through untouched, so one file can train decisions and extraction together.

In [ ]:
#| export
def gliner_example(record:dict) -> dict:
    "One GLiNER2 training example from a typed-decision record"
    rec = load_record(record, parse_state=False)
    qs, gold, cls = as_questions(rec["questions"]), rec.get("gold") or {}, []
    for qid, q in qs.items():
        t, label = q.target(gold.get(qid))
        if t is None: continue
        names, desc, keys = gliner_labels(q)
        key = q.keys[label]
        c = {"task": qid, "labels": names, "true_label": [names[keys.index(key)]], "prompt": q.instructions}
        if desc: c["label_descriptions"] = desc
        cls.append(c)
    out = {"classifications": cls} | {k: rec[k] for k in ("entities", "relations", "json_structures") if k in rec}
    return {"input": render_state(rec["state"]), "output": out}

def write_gliner_jsonl(records, path) -> Path:
    "A GLiNER2 training file from typed-decision records"
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        for r in records: f.write(json.dumps(gliner_example(r), ensure_ascii=False) + "\n")
    return path

In [ ]:
ex = gliner_example(recs[0])
[c | {"prompt": c["prompt"][:30]} for c in ex["output"]["classifications"]][:2]

[{'task': 'action',
  'labels': ['continue', 'human_review', 'observe', 'stop'],
  'true_label': ['human_review'],
  'prompt': 'What should the observability ',
  'label_descriptions': {'continue': 'Let the agent proceed without interruption.',
   'human_review': 'Queue this trace for a human to review.',
   'observe': 'Keep running, but flag the trace for later sampling.',
   'stop': 'Halt the agent now.'}},
 {'task': 'needs_review',
  'labels': ['yes', 'no'],
  'true_label': ['yes'],
  'prompt': 'This trace requires human revi',
  'label_descriptions': {'yes': 'A human should inspect this run.',
   'no': 'No human attention is warranted.'}}]

In [ ]:
c = {x["task"]: x for x in ex["output"]["classifications"]}
test_eq(c["action"]["true_label"], ["human_review"])
test_eq(c["needs_review"]["true_label"], ["yes"])
test_eq(c["urgency"]["true_label"], ["1"])
test_eq(json.loads(Path(write_gliner_jsonl(recs[:3], Path(tempfile.mkdtemp()) / "t.jsonl")).read_text().splitlines()[2])["input"][:1], "{")

## GlinerLearner

`GlinerLearner` writes the splits to JSONL and drives GLiNER2's `ExtractorTrainer`: `train="head"` is LoRA on the classification head alone, the frozen-encoder variant; `"lora"` adds the encoder's projections; `"full"` trains everything at GLiNER2's two learning rates (encoder 1e-5, heads 5e-4). `calibrate` fits one temperature per type and option-count bucket on the held-out slice, from the log-probabilities the model reports at temperature 1, and the decider applies them through the classifier's own temperature. `export` writes GLiNER2's format, a merged model or, with `merge=False`, the PEFT adapter of a LoRA run, plus `sysone_gliner.json` with the temperatures; `Decider.load` recognises both.

In [ ]:
#| export
_LORA_TARGETS = {"head": ["classification_head"], "lora": ["encoder", "classification_head"]}

class GlinerLearner:
    "Fine-tune a GLiNER2 checkpoint on typed decisions with GLiNER2's own trainer"
    def __init__(self, data:TypedDecisions, model:str="fastino/GLiNER2.5-Decide", train:str="head", path=None, **config):
        if train not in ("head", "lora", "full"): raise ValueError(f"train must be 'head', 'lora' or 'full', not {train!r}")
        self.data, self.base, self.train, self.path = data, model, train, Path(path or tempfile.mkdtemp(prefix="sysone-gliner-"))
        self.config, self.temperatures, self.model = config, {}, None
        self.classify = None   # the classify function; gliner2's constrained classifier on the model when None
    def __repr__(self): return f"GlinerLearner({self.base} · {self.train} · {len(self.temperatures)} temperatures)"

    def jsonl(self, split:str="train") -> Path | None:
        "The split as a GLiNER2 training file"
        cases = self.data.cases.get(split)
        if cases is None: return None
        return write_gliner_jsonl((load_record(cases[i]) for i in range(len(cases))), self.path / f"{split}.jsonl")

    def _load(self):
        if self.model is None:
            from gliner2 import AutoExtractor
            self.model = AutoExtractor.from_pretrained(self.base)
        return self.model

    def training_config(self, epochs:int=3, **kw):
        "GLiNER2's TrainingConfig for this regime"
        from gliner2.training.trainer import TrainingConfig
        lora = self.train in _LORA_TARGETS
        base = dict(output_dir=str(self.path / "trainer"), num_epochs=epochs, encoder_lr=1e-5, task_lr=5e-4, use_lora=lora,
                    save_adapter_only=lora, eval_strategy="epoch" if self.data.cases.get("valid") is not None else "no",
                    num_workers=0, seed=0) | ({"lora_target_modules": _LORA_TARGETS[self.train]} if lora else {})
        return TrainingConfig(**(base | self.config | kw))

    def fit(self, epochs:int=3, **kw) -> dict:
        "Train with ExtractorTrainer"
        from gliner2.training.trainer import ExtractorTrainer
        trainer = ExtractorTrainer(self._load(), self.training_config(epochs, **kw))
        valid = self.jsonl("valid")
        res = trainer.train(train_data=str(self.jsonl("train")), eval_data=str(valid) if valid else None)
        self.model = trainer.model
        return res

    def decider(self, calibrated:bool=True, **kw) -> GlinerDecider:
        "A `GlinerDecider` over the model in memory"
        return GlinerDecider(self._load(), self.temperatures if calibrated else {}, classify=self.classify, **kw)
    def predict(self, state, questions) -> dict: return self.decider().predict(state, questions)

In [ ]:
#| export
def _calibration_rows(decider, records):
    logits, targets, masks, qtypes = [], [], [], []
    for r in records:
        probs = decider.probabilities(r["state"], r["questions"])
        for qid, q in as_questions(r["questions"]).items():
            t, _ = q.target((r.get("gold") or {}).get(qid))
            if t is None: continue
            logits.append(np.log(np.clip(probs[qid], 1e-9, 1))); targets.append(t); qtypes.append(q.qtype)
    K = max(len(z) for z in logits)
    pad = lambda v, fill: np.array([np.pad(np.asarray(x, float), (0, K - len(x)), constant_values=fill) for x in v])
    return pad(logits, -1e4), pad(targets, 0.0), pad([np.ones(len(z)) for z in logits], 0).astype(bool), np.array(qtypes)

@patch
def calibrate(self:GlinerLearner, split:str="calib", min_rows:int=10) -> dict:
    "Temperatures per type and bucket from the log-probabilities (at temperature 1) on held-out rows"
    cases = self.data.cases.get(split)
    if cases is None: cases = self.data.cases.get("valid")
    if cases is None: raise ValueError("calibration needs held-out rows: a calib or valid split")
    z, t, m, q = _calibration_rows(self.decider(calibrated=False), [load_record(cases[i]) for i in range(len(cases))])
    self.temperatures = fit_temperatures(z, t, m, q, min_rows)
    return self.temperatures

@patch
def export(self:GlinerLearner, path, merge:bool=True) -> Path:
    "Write GLiNER2's format (merged weights, or the adapter with merge=False) and sysone_gliner.json"
    learn, path = self, Path(path); model = self._load()
    adapter = hasattr(model, "peft_config") and not merge
    if hasattr(model, "merge_and_unload") and merge: model = model.merge_and_unload()
    model.save_pretrained(str(path))
    (path / "sysone_gliner.json").write_text(json.dumps({"backend": "gliner2", "base": learn.base, "regime": learn.train,
                                                         "adapter": adapter, "temperature": learn.temperatures, "name": path.name}, indent=2))
    return path

Calibration works on any classify function, so the stand-in shows it: a classifier that is sure of the wrong answer half the time is told to soften:

In [ ]:
from sysone.data import TypedDecisions

In [ ]:
data = TypedDecisions.from_records(recs, calib=0)
gl = GlinerLearner(data, "stand-in")
def overconfident(text, tasks):
    return {t["task"]: {n: (0.97 if i == len(text) % len(t["names"]) else 0.03 / (len(t["names"]) - 1)) for i, n in enumerate(t["names"])}
            for t in tasks}
gl.model, gl.classify = object(), overconfident
gl.data.cases["calib"] = gl.data.cases["train"]
temps = gl.calibrate(min_rows=5)
temps

{'choice': 5.0,
 'choice:3-5': 5.0,
 'score': 5.0,
 'score:3-5': 5.0,
 'noul': 5.0,
 'noul:2': 5.0}

In [ ]:
assert all(v > 1 for v in temps.values())
test_eq(Path(gl.jsonl("train")).exists(), True)

The one-call learner, and the Decide checkpoint end to end (needs `sysone[gliner]` and the 1.9 GB checkpoint):

In [ ]:
#| export
def decision_learner(data:TypedDecisions, model:str="fastino/GLiNER2.5-Decide", train:str="head", **config) -> GlinerLearner:
    "A `GlinerLearner`: LoRA on the classification head (`head`), plus the encoder (`lora`), or everything (`full`)"
    return GlinerLearner(data, model, train, **config)

In [ ]:
#| eval: false
data  = TypedDecisions.from_hub("LocalLLaMA/typed-decisions", valid="test")
learn = decision_learner(data, "fastino/GLiNER2.5-Decide", train="head")
learn.fit(3)                     # ExtractorTrainer: LoRA on the classification head, task_lr 5e-4
learn.calibrate()                # a temperature per bucket, applied through the classifier's own temperature
learn.export("decide-ft")        # GLiNER2 format + sysone_gliner.json

from sysone.inference import Decider
jev = Decider.load("decide-ft")   # a GlinerDecider
jev.predict(state, questions)
text = "Invoice 4411 was paid twice by Acme Ltd; contact Dana Reyes for the refund."
jev.model.extract_entities(text, ["invoice number", "company", "person"])       # gliner2, untouched

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()